# Conference Evaluation: Spiking U-Net vs Standard U-Net (BraTS 2023)

This notebook evaluates the **Agentic Spiking Transformer U-Net** against standard baseline metrics to demonstrate its hardware efficiency and routing efficacy.

### Metrics Covered:
1. **Semantic Accuracy (3D Dice Score):** Evaluates segmentation overlap for WT (Whole Tumor), TC (Tumor Core), and ET (Enhancing Tumor).
2. **Computational Complexity (FLOPs vs. SOPs):** Compares traditional Floating Point Operations vs Synaptic Operations (Additions) driven by the Spiking Tokenizer's firing rates.
3. **Theoretical Energy Savings:** Translates FLOPs ($4.6$ pJ) and SOPs ($0.9$ pJ) into theoretical Joules per inference.
4. **Agentic Routing Savings:** Calculates the exact energy saved across a dataset validation split by bypassing $T=4$ using the Phi-3-Mini Orchestrator.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Evaluation Device: {device}')

Evaluation Device: cpu


## 1. 3D Dice Score Metric

In [2]:
def calculate_dice_score(preds, targets, smooth=1e-5):
    """
    Calculates the 3D multi-class Dice Similarity Coefficient.
    preds: Activation logits [B, C, D, H, W]
    targets: Binary Ground truth [B, C, D, H, W]
    Returns: WT, TC, and ET Dice scores.
    """
    preds = torch.sigmoid(preds)
    preds = (preds > 0.5).float()
    
    dice_scores = []
    for c in range(targets.shape[1]): # Iterate over WT, TC, ET
        pred_c = preds[:, c, ...]
        target_c = targets[:, c, ...]
        
        intersection = (pred_c * target_c).sum(dim=(1, 2, 3))
        union = pred_c.sum(dim=(1, 2, 3)) + target_c.sum(dim=(1, 2, 3))
        
        dice = (2. * intersection + smooth) / (union + smooth)
        dice_scores.append(dice.mean().item())
        
    return {'WT': dice_scores[0], 'TC': dice_scores[1], 'ET': dice_scores[2]}

### Dummy Test Case (Replace `output_tensor` and `target_tensor` with actual validation loop tensors) ###
dummy_preds = torch.randn(1, 3, 64, 64, 64)
dummy_targets = torch.randint(0, 2, (1, 3, 64, 64, 64)).float()
dice = calculate_dice_score(dummy_preds, dummy_targets)
print(f"Validation Dice Score:\nWhole Tumor (WT): {dice['WT']:.4f} | Tumor Core (TC): {dice['TC']:.4f} | Enhancing Tumor (ET): {dice['ET']:.4f}")

Validation Dice Score:
Whole Tumor (WT): 0.5000 | Tumor Core (TC): 0.5023 | Enhancing Tumor (ET): 0.5009


## 2. Theoretical Computational Complexity & Hardware Energy
Based on standard 45nm CMOS process assumptions:
- MAC (Multiply-Accumulate, typical ANN): $4.6$ $pJ$
- AC (Accumulate, SNN synaptic routing): $0.9$ $pJ$

In [3]:
class HardwareEnergyEvaluator:
    def __init__(self, mac_energy_pj=4.6, ac_energy_pj=0.9):
        self.e_mac = mac_energy_pj
        self.e_ac = ac_energy_pj

    def calculate_snn_energy(self, total_params, firing_rate, T):
        """
        SNNs bypass multipliers. Energy relies mostly on Accumulations driven by the sparsity (firing rate).
        SOPs = Synaptic Operations = FLOPs * Firing_Rate
        """
        sops = total_params * firing_rate * T
        energy_pj = sops * self.e_ac
        return sops, energy_pj

    def calculate_ann_energy(self, total_params):
        """
        Standard ANNs run full MAC ops at T=1 for every parameter.
        """
        flops = total_params
        energy_pj = flops * self.e_mac
        return flops, energy_pj

evaluator = HardwareEnergyEvaluator()

# Example: 3D Patch Convolution Block (E.g. 32 channels, 3x3x3 kernel over 64x64x64 volume)
spatial_dims = 64 * 64 * 64
kernel_ops = 32 * 32 * 3 * 3 * 3 # C_in * C_out * K_d * K_h * K_w
total_block_ops = spatial_dims * kernel_ops

# Assuming Spiking Tokenizer yields a ternary firing rate (sparsity) of 12%
T_high = 4
firing_rate = 0.12 

ann_flops, ann_energy = evaluator.calculate_ann_energy(total_block_ops)
sops, snn_energy = evaluator.calculate_snn_energy(total_block_ops, firing_rate, T=T_high)

print(f"--- Computational Complexity (Single Block Layer) ---")
print(f"Standard ANN: {ann_flops/1e9:.2f} GFLOPs -> Energy: {ann_energy/1e9:.2f} mJ")
print(f"Spiking NN (T={T_high}): {sops/1e9:.2f} GSOPs -> Energy: {snn_energy/1e9:.2f} mJ")
print(f"Hardware Energy Reduction per block: {(1 - (snn_energy / ann_energy)) * 100:.2f}%")


--- Computational Complexity (Single Block Layer) ---
Standard ANN: 7.25 GFLOPs -> Energy: 33.34 mJ
Spiking NN (T=4): 3.48 GSOPs -> Energy: 3.13 mJ
Hardware Energy Reduction per block: 90.61%


## 3. Agentic Routing Triage Savings
Simulating the mathematical savings across a dataset of $N$ validation volumes when Phi-3 intercepts complex patches.

In [4]:
def calculate_triage_savings(N_total, T1_energy_mJ, T4_energy_mJ, agent_routed_low_ratio):
    """
    agent_routed_low_ratio: Percentage of patches Phi-3 decided were simple enough for T=1.
    """
    baseline_snn_energy = N_total * T4_energy_mJ
    
    routed_low_count = int(N_total * agent_routed_low_ratio)
    routed_high_count = N_total - routed_low_count
    
    agent_energy = (routed_low_count * T1_energy_mJ) + (routed_high_count * T4_energy_mJ)
    
    savings = (1 - (agent_energy / baseline_snn_energy)) * 100
    return agent_energy, baseline_snn_energy, savings

# Simulation Metrics base on our single block
T1_energy = snn_energy / 1e9 / 4.0 # Roughly dividing T=4 by 4
T4_energy = snn_energy / 1e9
N_val_patches = 1000

# Assume Phi-3 routes 65% of background/simple patches to T=1 Low Power Mode
routing_ratio = 0.65

agent_E, base_E, saved = calculate_triage_savings(N_val_patches, T1_energy, T4_energy, routing_ratio)

print(f"--- Dataset Inference Energy (N={N_val_patches} Patches) ---")
print(f"Baseline Spiking (Static T=4): {base_E:.2f} mJ")
print(f"Agentic Spiking (Dynamic): {agent_E:.2f} mJ")
print(f"LLM Orchestrator System-level Energy Savings: {saved:.2f}%")


--- Dataset Inference Energy (N=1000 Patches) ---
Baseline Spiking (Static T=4): 3131.03 mJ
Agentic Spiking (Dynamic): 1604.65 mJ
LLM Orchestrator System-level Energy Savings: 48.75%
